# 03C 特征重要性与模型解释：模型学到了什么？

> 🟢 **Level A · 必须掌握** | 完成标准：比较至少两种 importance 方法，并明确 predictive association ≠ causality。

本节依次学习 `correlation → RF impurity importance → permutation importance → SHAP`。

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
df=pd.read_csv('https://raw.githubusercontent.com/gokhanonderaksu/COFSpace/main/OnlyCoRECOF%20-%20Feature%20Sets/CoRECOF%20-%20CO2%20-%201%20BAR.csv')
target='CO2-1 bar (mol/kg)'
features=['PLD (Å)','LCD (Å)','Sacc (m2/g-1)','Porosity','%C','%H','%N','%O','%Metalloid','%Halogen','%Ametal']
X=df[features].fillna(df[features].median()); y=df[target]
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,random_state=42)
rf=RandomForestRegressor(n_estimators=400,random_state=42,n_jobs=-1).fit(Xtr,ytr)


In [ ]:
display(df[features+[target]].corr(numeric_only=True)[target].drop(target).sort_values(key=abs,ascending=False).to_frame('Pearson r'))
pd.Series(rf.feature_importances_,index=features).sort_values().plot.barh(); plt.xlabel('RF impurity importance'); plt.show()
perm=permutation_importance(rf,Xte,yte,n_repeats=20,random_state=42,scoring='r2')
pd.Series(perm.importances_mean,index=features).sort_values().plot.barh(); plt.xlabel('Permutation importance'); plt.show()


In [ ]:
!pip -q install shap
import shap
sample=Xte.iloc[:200]
shap.summary_plot(shap.TreeExplainer(rf).shap_values(sample),sample)


## 回到 COF 化学
不要停在 importance 排名。继续问低压 adsorption 是否更受 affinity/chemistry 控制，高压是否更受 pore volume/ASA 控制，以及 PLD/LCD 相关性是否分摊 importance。
